# Session 28 Solutions: Imbalanced Data

Data Science with Python, Level 2, Stage 4: Applied ML

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             precision_recall_curve, average_precision_score, roc_auc_score)

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
tx = pd.read_csv(DATA_URL + "card_transactions.csv")
cols = ["amount", "hour", "online"]
X = tx[cols].astype(float)
y = tx["is_fraud"]

## In-Class Exercises

### In-Class Exercise 1: Weights for Boosting

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
for w in [None, "balanced"]:
    h = HistGradientBoostingClassifier(class_weight=w, learning_rate=0.05, max_depth=3, random_state=0).fit(X_train, y_train)
    print(w, confusion_matrix(y_test, h.predict(X_test)).ravel(), round(average_precision_score(y_test, h.predict_proba(X_test)[:, 1]), 3))

None [1469    4   12   15] 0.601
balanced [1375   98    6   21] 0.627


### In-Class Exercise 2: Cost of Mistakes

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
proba = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train).predict_proba(X_test)[:, 1]
for t in [0.5, 0.2, 0.1, 0.05, 0.02]:
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= t).astype(int)).ravel()
    print(t, fn * 270 + fp * 5)

0.5 3800
0.2 3295
0.1 3060
0.05 2350
0.02 1680


### In-Class Exercise 3: Custom Weights

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
for w in [1, 5, 20, 55]:
    p = make_pipeline(StandardScaler(), LogisticRegression(class_weight={0: 1, 1: w})).fit(X_train, y_train).predict(X_test)
    print(w, round(recall_score(y_test, p), 2), round(precision_score(y_test, p), 2))

1 0.48 0.76
5 0.56 0.54
20 0.7 0.28
55 0.89 0.15


### In-Class Exercise 4: All the Features

In [5]:
full = tx[["amount", "hour", "online", "distance_from_home_km", "new_merchant"]].astype(float)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
m = make_pipeline(StandardScaler(), LogisticRegression())
print(round(cross_val_score(m, full, y, cv=cv, scoring="average_precision").mean(), 3))

0.995


## Part A: Basic

### Exercise 1: Imbalance Ratio

In [6]:
print("churn", round((pd.read_csv(DATA_URL + "telecom_churn.csv")["churn"] == "Yes").mean(), 3))
print("fraud", round(pd.read_csv(DATA_URL + "card_transactions.csv")["is_fraud"].mean(), 3))
print("passed", round(pd.read_csv(DATA_URL + "exam_scores.csv")["passed"].mean(), 3))
# Fraud is strongly imbalanced; churn is moderately imbalanced.

churn 0.303
fraud 0.018
passed 0.45


### Exercise 2: What Balanced Means

In [7]:
from sklearn.utils.class_weight import compute_class_weight
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
n = len(y_train)
print(n / (2 * (y_train == 0).sum()), n / (2 * (y_train == 1).sum()))
print(compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train))

0.5091649694501018 27.77777777777778
[ 0.50916497 27.77777778]


## Part B: Intermediate

### Exercise 3: Threshold from Validation

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=0, stratify=y_train)
m = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_fit, y_fit)
prec, rec, thr = precision_recall_curve(y_val, m.predict_proba(X_val)[:, 1])
ok = rec[:-1] >= 0.8
best = thr[np.argmax(np.where(ok, prec[:-1], -1))]
p = (m.predict_proba(X_test)[:, 1] >= best).astype(int)
print(round(best, 3), round(recall_score(y_test, p), 2), round(precision_score(y_test, p), 2))

0.012 0.89 0.12


### Exercise 4: Fraud by Category

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
print(tx.groupby("category")["is_fraud"].mean().round(3).sort_values())
cv = StratifiedKFold(5, shuffle=True, random_state=0)
pre = ColumnTransformer([("num", StandardScaler(), cols), ("cat", OneHotEncoder(), ["category"])])
with_cat = make_pipeline(pre, LogisticRegression())
print("with", round(cross_val_score(with_cat, tx[cols + ["category"]], y, cv=cv, scoring="average_precision").mean(), 3))
print("without", round(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X, y, cv=cv, scoring="average_precision").mean(), 3))

category
Restaurants    0.016
Electronics    0.017
Fuel           0.017
Clothing       0.019
Groceries      0.019
Travel         0.020
Name: is_fraud, dtype: float64
with 0.655
without 0.659


## Part C: Challenge

### Exercise 5: Night-Time Feature

Good features often help more than any balancing trick. Session 31 covers feature engineering in depth.

In [10]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)
d = tx.copy()
d["night"] = ((d["hour"] >= 22) | (d["hour"] < 6)).astype(int)
d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
for name, c in [("hour", ["amount", "online", "hour"]), ("night", ["amount", "online", "night"]), ("cyclic", ["amount", "online", "hour_sin", "hour_cos"])]:
    m = make_pipeline(StandardScaler(), LogisticRegression())
    print(name, round(cross_val_score(m, d[c].astype(float), y, cv=cv, scoring="average_precision").mean(), 3))

hour 0.659
night 0.664
cyclic 0.656


## Part D: Cumulative Project: Customer Churn

### Exercise 6: Customer Churn: Step 5, Recall Matters

In [11]:
from sklearn.model_selection import cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
data = pd.read_csv(DATA_URL + "telecom_churn.csv")
yc = (data["churn"] == "Yes").astype(int)
Xc = data.drop(columns=["customer_id", "churn"])
X_dev, X_final, y_dev, y_final = train_test_split(Xc, yc, test_size=0.2, stratify=yc, random_state=0)
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls"]
cat_cols = ["gender", "contract", "internet", "payment_method"]
bool_cols = ["senior", "paperless_billing"]
def preprocess():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore")), cat_cols),
        ("bool", "passthrough", bool_cols),
    ])
for w in [None, "balanced"]:
    m = make_pipeline(preprocess(), LogisticRegression(class_weight=w, max_iter=1000))
    r = cross_validate(m, X_dev, y_dev, cv=5, scoring=["recall", "precision", "roc_auc"])
    print(w, {k: round(v.mean(), 3) for k, v in r.items() if k.startswith("test_")})
# The balanced model reaches the 70 percent recall goal; precision drops, which is acceptable for a cheap retention call.

None {'test_recall': np.float64(0.431), 'test_precision': np.float64(0.627), 'test_roc_auc': np.float64(0.786)}
balanced {'test_recall': np.float64(0.769), 'test_precision': np.float64(0.508), 'test_roc_auc': np.float64(0.786)}


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | Compare with the majority baseline. |
| 2 | B | Rare class errors get a larger weight. |
| 3 | B | The test set must stay realistic. |
| 4 | B | They focus on the positive class. |
| 5 | A | More cases are flagged. |